# 车道线识别 01

## 使用的数据库

https://www.kaggle.com/datasets/manideep1108/tusimple

TuSimple 数据集包含 6,408 张美国高速公路的道路图像，分辨率为 1280×720。

该数据集由 3,626 张训练集图像、358 张验证集图像和 2,782 张测试集图像组成，其中测试集图像涵盖了不同的天气条件。

这是包含额外训练分割标注的完整版 TuSimple 数据集。

In [16]:
#  下载数据集，数据集大小为21G，可以在 kaggle 上在线运行

# import kagglehub

# path = kagglehub.dataset_download("manideep1108/tusimple")

# print("Path to dataset files:", path)

## U-Net 核心工作原理

U-Net 本质是为像素级图像分割设计的 “编码器 - 解码器 + 跳跃连接” 架构，所有的设计决策都围绕 “解决分割任务的核心痛点：既要全局语义，又要精准像素定位”， 尤其适配车道线这种
“细，长，易断, 依赖细节” 的分割目标

普通卷积神经网络（如VGG，ResNet) 的设计目标是“图像分类” ---- 只需要输出“这张图里有没有车道线”

但车道线识别是分割任务，需要输出“每个像素是不是车道线”，普通的CNN存在两个致命问题

1. 下采样丢失细节：CNN通过池化下采样压缩尺寸，提取全局特征，但会丢失车道线的细轮廓，断点，边缘等关键细节
2. 上采样无法还原细节：如果直接对全局特征上采样恢复尺寸，得到的车道线会模糊，断连，无法精准定位
3. 全连接层破坏空间信息：分类网络的全连接层会把特征展平为一维向量，完全丢失像素的空间位置关系（比如这个像素在车道线左侧还是右侧）

U-Net 的所有设计，都是为了同时保留“全局语义特征”和“像素级细节特征”，完美的解决上述问题，这也是他比普通CNN更适合车道线分割的核心原因

## U-Net 核心工作流程(分4步，对应结构设计）

U-Net的工作过程可以拆分为“压缩-融合-恢复-输出”四步，每一步的操作和设计目的都针对车道线分割优化

### 第一步：编码器（Encoder）-- 压缩图像，提取”全局语义特征“ (左半部分)

核心操作：卷积 + 池化，层层下采样

假设输入一张车道线图像（比如 640x360x3）,经过4轮（两次3x3卷积 + 1次2x2最大池化）

具体动作

+ 第一轮：640x360x3 -> 640x360x64 (卷积提取车道线边缘/纹理) -> 320x180x64(池化压缩)
+ 第二轮：320x180x64 -> 320x180x128 (提取车道线局部连续性) -> 160x90x128(池化)
+ 第三轮：160x90x128 -> 160x90x256 (提取车道线走向) -> 80x45x256(池化)
+ 第四轮：80x45x256 -> 80x45x512 (提取全局语义: 区分车道线，路面，障碍物) -> 40x22x512(池化)
+ 最后经过”瓶颈层“：40x22x512 -> 40x22x1024 (融合所有层级的全局特征)

**为什么这么设计？**

1. 3x3小卷积核：比大核（如7x7）更适配车道线的细线条特征，避免模糊车道线边缘
2. 两次卷积+ReLU：通过非线性变换，让网络学习车道线的复杂特征（比如虚线/实线/弯道的差异）
3. 最大池化（而非平均池化）：保留车道线的”关键像素“（比如车道线的边缘点），平均池化会平滑掉这些细节
4. 下采样（尺寸减半，通道翻倍）：用”小尺寸，高通道“的特征图存储全局语义，减少计算量，同时让网络关注”车道线整体走向“而非”单个像素“

### 第二步：跳跃连接(Skip Connection) -- 融合“全局特征 + 细节特征" (U型的横杠)

核心操作：编码器特征直接拼接到解码器对应层

具体动作

+ 解码器每一层输出的特征图（比如第一轮的640x360x64, 第二轮的320x180x128）,会被“原样保存”，在解码器对应层上采样后，拼接到解码器特征图上
+ 比如，编码器第四层（80x45x512）-> 拼接至解码器第四层（80x45x512）-> 融合后 80x45x1024

**为什么这么设计**

1. 解决细节丢失问题：编码器浅层（第1/2层）保存了车道线的“像素级细节”（比如虚线的断点，磨损车道线的残迹），解码器仅靠上采样无法恢复这些细节，拼接能直接把细节“喂”给解码器
2. 拼接而非相加：车道线特征稀疏，相加会丢失细节（比如 ResNet 的残差连接），拼接能保留所有细节特征（通道数翻倍），让解码器同时拥有“全局语义（知道这是车道线）”和“细节特征（知道车道线的每个像素在哪）”
3. 适配车道线的细线条特征：哪怕车道线只有1-2像素宽，拼接的细节特征也能让编码器精准定位，避免分割出“宽模糊”的车道线

### 第三步：解码器（Decoder）--- 恢复尺寸，还原”像素级位置“（右半部分）

核心操作：上采样 + 卷积 + 拼接，层层恢复尺寸

具体动作

从 瓶颈层 的 40x22x1024 特征图开始吗，每一层都进行上采样恢复尺寸，**具体动作**

+ 解码器每一层输出的特征图（比如第一轮的640x360x64, 第二轮的320x180x128）,会被“原样保存”，在解码器对应层上采样后，拼接到解码器特征图上
+ 比如，编码器第四层（80x45x512）-> 拼接至解码器第四层（80x45x512）-> 融合后 80x45x1024

**为什么这么设计**

1. 解决细节丢失问题：编码器浅层（第1/2层）保存了车道线的“像素级细节”（比如虚线的断点，磨损车道线的残迹），解码器仅靠上采样无法恢复这些细节，拼接能直接把细节“喂”给解码器
2. 拼接而非相加：车道线特征稀疏，相加会丢失细节（比如 ResNet 的残差连接），拼接能保留所有细节特征（通道数翻倍），让解码器同时拥有“全局语义（知道这是车道线）”和“细节特征（知道车道线的每个像素在哪）”
3. 适配车道线的细线条特征：哪怕车道线只有1-2像素宽，拼接的细节特征也能让编码器精准定位，避免分割出“宽模糊”的车道线

### 第三步：解码器（Decoder）--- 恢复尺寸，还原”像素级位置“（右半部分）

核心操作：上采样 + 卷积 + 拼接，层层恢复尺寸

**具体动作**

从 瓶颈层 的 40x22x1024 特征图开始吗，每一层都进行上采样恢复尺寸

**为什么这么设计？**

1. 转置卷积（UpConv）上采样：步长2，精准恢复到编码器对应层的尺寸，避免车道线位置偏移（比如用插值上采样会导致位置错位）
2. 拼接后再卷积：融合全局特征和细节特征后，通过卷积整合信息（比如区分”车道线细节“和”路面噪音细节“）
3. 通道数逐层减半：从高通道（全局特征）逐步过渡到地通道（像素级特征），最终恢复到输入图像的尺寸，保证每个像素都有对应的特征输出

### 第四步: 输出层 -- 像素级分类，输出车道线掩码

核心操作： 1x1卷积 + 激活函数，输出分割结果

**具体动作**

解码器最后一层输出 640x360x64 的特征图，经过1x1卷积将通道数从64压缩为1（车道线二分类： 1 = 车道线，0 = 非车道线），再通过sigmod 激活函数将输出映射到0-1之间，最终输出 640x360x1 车道线掩码

**为什么这么设计？**

1. 1x1卷积：不改变图像尺寸，仅调整通道数，避免模糊车道线的像素位置
2. sigmod 激活函数：将输出映射到0-1之间，方便后续处理（比如二分类）；如果是多类车道线（左/中/右），改用 softmax + 通道数 = 类别数
3. 无全连接层：全连接层会展平特征，丢失空间位置信息，U-Net 全程用卷积操作，保留所有像素的空间关系，这是分割任务的关键

## 代码实现

### 所需的头文件

In [17]:
## 所需的头文件及其作用

import json                             # 用于解析 / 生成 JSON 格式数据
import os                               # 用于操作系统交互（文件 / 路径 / 环境变量）

import cv2                              # 计算机视觉核心库（图像读取 / 处理 / 特征提取）
import matplotlib.pyplot as plt         # 绘图可视化库，用于绘制图表 / 图像

import numpy as np                      # 数值计算核心库，处理多维数组（矩阵）

from sklearn.cluster import DBSCAN      # 无监督聚类算法（密度聚类）
from sklearn.model_selection import train_test_split

import torch                            # PyTorch 核心库，构建 / 训练深度学习模型（张量计算 / GPU 加速）
import torch.nn as nn                   # PyTorch 神经网络模块，提供层 / 模型 / 损失函数的基础类
from torch.nn.modules.loss import _Loss # 损失函数基类（自定义损失函数时继承）
from torch.autograd import Variable
import torch.nn.functional as F

import tqdm                             # 进度条库，可视化循环 / 迭代的进度
import seaborn as sns                   # 基于 matplotlib 的高级可视化库，专注统计绘图

In [18]:
## 尝试打印torch的版本和CPU/GPU信息
print("Torch version : {}".format(str(torch.__version__)))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device : {}".format(device))

Torch version : 2.5.1
Torch device : cpu


In [19]:
## Kaggle 设置数据来源
macos_path = "/Volumes/GuanggangStorage/Dataset/kagglehub/datasets/manideep1108/tusimple/versions/5"
kaggle_path = "/kaggle/input/tusimple"
data_path = macos_path

train_data_path = data_path + "/TUSimple/train_set"
train_label_file1 = data_path + "/TUSimple/train_set/label_data_0313.json"
train_label_file2 = data_path + "/TUSimple/train_set/label_data_0531.json"
train_label_file3 = data_path + "/TUSimple/train_set/label_data_0601.json"

test_data_path = "/kaggle/input/tusimple/TUSimple/test_set"
test_label_file = "/kaggle/input/tusimple/TUSimple/test_set/test_tasks_0627.json"

## Macos 设置数据来源

# 数据过大，谨慎运行
# train_label_list = [train_label_file1, train_label_file2, train_label_file3]
# 运行 0601 数据
train_label_list = [train_label_file3]

In [20]:
## 读取数据函数
def preprocess_data(data_path, label_list, img_size = (256, 256)):
    images = []
    masks = []
    for label_path in label_list:
        with open(label_path, 'r') as f:
            labels = [json.loads(line) for line in f]
            for label in labels:
                img_path = os.path.join(data_path, label["raw_file"])
                img = cv2.imread(img_path)
                img = cv2.resize(img, img_size)
                images.append(img)

                # 建立一个纯黑的图片
                mask = np.zeros((720, 1280), dtype = np.uint8)
                lanes = label['lanes']
                h_samples = label['h_samples']
                for lane in lanes:
                    if len(lane) > 0:
                        points = [(x, y) for x, y in zip(lane, h_samples) if x > 0]
                        for i in range(len(points) - 1):
                            # 宽度 为 5，颜色为255
                            cv2.line(mask, points[i], points[i+1], 255, 5)
                mask = cv2.resize(mask, img_size)
                masks.append(mask)
    images = np.array(images)
    masks = np.array(masks)
    return images, masks

images, masks = preprocess_data(train_data_path, train_label_list)

x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(images, masks, test_size=0.2, random_state=40)

## 归一化数据
x_train = x_train_raw / 255.0
y_train = y_train_raw / 255.0
x_test = x_test_raw / 255.0
y_test = y_test_raw / 255.0

## 这里因为 y_train 和 y_test 的结果都是单通道的，所以需要再后面再加一个通道
print("Y Shape: {}".format(y_train.shape))
y_train = np.expand_dims(y_train, axis=-1)
y_test = np.expand_dims(y_test, axis=-1)
print("Y Shape: {}".format(y_train.shape))

Y Shape: (328, 256, 256)
Y Shape: (328, 256, 256, 1)


In [21]:
## 将数据转化成能够由 nn.conv 的格式，方便后续处理
from torch.utils.data import TensorDataset, DataLoader

x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

## permute等价于transpose，支持多维度交换
x_train_tensor = x_train_tensor.permute(0, 3, 1, 2)
y_train_tensor = y_train_tensor.permute(0, 3, 1, 2)

x_test_tensor = x_test_tensor.permute(0, 3, 1, 2)
y_test_tensor = y_test_tensor.permute(0, 3, 1, 2)

## 创建数据库, 使x和y一一对应，方便后续加载，打乱，并行读取等操作
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

## batch_size 表示一次加载数据时的长度，防止数据溢出
# shuffle 表示在每次训练时是否有必要随机打乱数据顺序
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# 缓存数据 到 IPython
%store train_loader
%store test_loader

Stored 'train_loader' (DataLoader)
Stored 'test_loader' (DataLoader)


In [22]:
## U-Net的实现

## 使用 Bottleneck 减少计算量，具体可以搜以下 bottleneck 的核心理念
class Bottleneck(nn.Module):
    ## stride 1 不降低 图像大小, 使用 maxpool 进行降采样
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super(Bottleneck, self).__init__()

        # 先降低通道，减少计算量, 如果小于4 就不降低了
        if out_channels >= 4:
            inter_channels = out_channels // 4
        else:
            inter_channels = out_channels
        # 尺寸计算公式 h_out = floor(h_in + 2*padding - k - 2) / s + 1
        # 主路径
        # h_in = h_out
        self.conv1 = nn.Conv2d(in_channels, inter_channels, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(inter_channels)

        # h_out = (h_in + 2 - 3 - 2) / 2 + 1
        self.conv2 = nn.Conv2d(inter_channels, inter_channels, kernel_size, 
                               stride, padding=1, bias=False)                    
        self.bn2 = nn.BatchNorm2d(inter_channels)
        
        self.conv3 = nn.Conv2d(inter_channels, out_channels, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                # 1x1 卷积同时处理：通道数变化 + 空间下采样
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))

        # 如果尺寸不一致, 需要调整尺寸
        if self.downsample is not None:
            identity = self.downsample(identity)
        
        x += identity
        x = self.relu(x)
        return x

class EncodeLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super(EncodeLayer, self).__init__()
        self.bottleneck = Bottleneck(in_channels, out_channels, kernel_size, stride)
        self.maxpool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        # 图像尺寸不变，通道数增加
        self.res = self.bottleneck(x)
        # 降采样，图像尺寸减半
        x = self.maxpool(self.res)
        return x

    def get_resident(self):
        return self.res

# DecodeLayer 有两个特殊要求
#   1. 在最底层时的卷积需要扩大通道
#   2. 在最上层时的上采样不要改变图像大小
class DecodeLayer(nn.Module):
    def __init__(self, in_channels, out_channels, conv_kernel=3, conv_stride=1, trans_kernel=4, trans_stride=2, trans_padding=1):
        super(DecodeLayer, self).__init__()
        if in_channels != out_channels:
            # 如果整体的通道数保持不变，就需要先将通道数扩大再缩小
            inter_channels = out_channels
        else:
            # 在最底层需要先把通道扩大一倍再缩小，才能使用之后的上采样拼接
            inter_channels = in_channels * 2
        
        self.bottleneck = Bottleneck(in_channels, inter_channels, conv_kernel, conv_stride)
        # 上采样图像尺寸放大一倍，通道缩小一倍
        self.conv_trans = nn.ConvTranspose2d(inter_channels,
                                             out_channels,
                                             kernel_size = trans_kernel,
                                             stride = trans_stride,
                                             padding = trans_padding)

    def forward(self, x):
        # 通道数减半，图像尺寸不变
        x = self.bottleneck(x)
        # 通道数减半，图像尺寸翻倍
        x = self.conv_trans(x)
        return x

class CustomUNet(nn.Module):
    def __init__(self, num_classes = 1):
        super(CustomUNet, self).__init__()

        # Encoder 编码器
        self.encoder1 = EncodeLayer(3, 64)       # res [b, 64, 1024, 1024]
        self.encoder2 = EncodeLayer(64, 128)     # res [b, 128, 512, 512]
        self.encoder3 = EncodeLayer(128, 256)
        self.encoder4 = EncodeLayer(256, 512)
        self.encoder5 = EncodeLayer(512, 1024)

        # Decoder 解码器
        self.decoder1 = DecodeLayer(1024, 1024)
        self.decoder2 = DecodeLayer(2048, 512)
        self.decoder3 = DecodeLayer(1024, 256)
        self.decoder4 = DecodeLayer(512, 128)
        self.decoder5 = DecodeLayer(256, 64)
        self.decoder6 = DecodeLayer(128, 1, trans_kernel=3, trans_stride=1, trans_padding=1)

        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        ## 假设 x [b, 3, 1024, 1024]
        x = self.encoder1(x)                     # out [b, 64, 512, 512] res [b, 64, 1024, 1024]
        x = self.encoder2(x)                     # out [b, 128, 256, 256] res [b, 128, 512, 512]
        x = self.encoder3(x)                     # out [b, 256, 128, 128] res [b, 256, 256, 256]
        x = self.encoder4(x)                     # out [b, 512, 64, 64] res [b, 512, 128, 128]
        x = self.encoder5(x)                     # out [b, 1024, 32, 32] res [b, 1024, 64, 64]
        
        x = self.decoder1(x)                     # out [b, 1024, 64, 64]
        x = self.crop_and_concat(x, self.encoder5.get_resident())  # out [b, 2048, 64, 64]
        x = self.decoder2(x)                     # out [b, 512，128， 128]
        x = self.crop_and_concat(x, self.encoder4.get_resident())  # out [b, 1024, 128, 128]
        x = self.decoder3(x)                     # out [b, 256, 256, 256]
        x = self.crop_and_concat(x, self.encoder3.get_resident())  # out [b, 512, 256, 256]
        x = self.decoder4(x)                     # out [b, 128, 512, 512]
        x = self.crop_and_concat(x, self.encoder2.get_resident())  # out [b, 256, 512, 512]
        x = self.decoder5(x)                     # out [b, 64, 1024, 1024]
        x = self.crop_and_concat(x, self.encoder1.get_resident())  # out [b, 128, 1024, 1024]
        x = self.decoder6(x)                     # out [b, 1, 1024, 1024]
        x = self.sigmoid(x)
        return x

    @staticmethod
    def crop_and_concat(x1, x2):
        h1, w1 = x1.shape[2], x1.shape[3]
        h2, w2 = x2.shape[2], x2.shape[3]

        # 计算差值
        delta_h = h2 - h1
        delta_w = w2 - w1

        # 对 拼接 进行中心裁剪
        x2_cropped = x2[:, :,
                        delta_h // 2 : delta_h // 2 + h1,
                        delta_w // 2 : delta_w // 2 + w1]

        # 拼接通道
        return torch.cat([x1, x2_cropped], dim = 1)

In [23]:
## 实例化模型
from torch.nn.parallel import DataParallel

# 自动选择设备
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
device_count = torch.cuda.device_count()

model = CustomUNet()

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = DataParallel(model, device_ids=[0, 1])

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
# model = model.to(device)

Using device: mps


In [24]:
## 训练


epochs = 150

results = {
    'epoch': [],
    'train_loss': [],
    'train_accuracy': [],
    'test_loss': [],
    'test_accuracy': []
}

for epoch in range(epochs):
    running_loss = 0
    model.train()

    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()

        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        _, predicted = torch.max(y, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            output = model(x)
            loss = criterion(output, y)
            _, predicted = torch.max(output, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    test_loss /= total
    test_acc = correct / total

    results['epoch'].append(epoch + 1)
    results['train_loss'].append(train_loss)
    results['train_accuracy'].append(train_acc)
    results['test_loss'].append(test_loss)
    results['test_accuracy'].append(test_acc)

    print(f"epoch {epoch+1}/{epochs} finished, running loss {running_loss}")
        

epoch 1/150 finished, running loss 129.31812524795532


KeyboardInterrupt: 

In [ ]:
## 保存和评估
import json

model_name = "CustomUNet_0601.pth"
result_name = "CustomUNet_0601_Result.json"

torch.save(model, model_name)

with open(result_name, 'w') as f:
    json.dump(results, f, indent=2)  # indent=2 让 JSON 格式化输出，便于阅读

print(f"{model_name} save success")

acc


In [ ]:
# import torchou
# import torch.nn as nn

# conv = nn.Conv2d(
#     in_channels = 1,
#     out_channels = 1,
#     kernel_size = 2,
#     stride = 2,
#     padding = 1,
#     bias = False
# )

# conv_trans = nn.ConvTranspose2d(
#     in_channels = 1,
#     out_channels = 1,
#     kernel_size = 2,
#     stride = 2,
#     padding = 0,
#     bias = False
# )

# conv.weight.data = torch.tensor([[[[1, 0], [0, 2]]]], dtype=torch.float32)
# conv_trans.weight.data = torch.tensor([[[[1, 0], [0, 2]]]], dtype=torch.float32)

# x = torch.tensor([[[[1, 2, 3], [4, 5, 6], [7, 8, 9]]]], dtype=torch.float32)

# ct_output = conv_trans(x)
# c_output = conv(x)

# print("输入特征图形状：", x.shape)  # torch.Size([1, 1, 2, 2])
# print("ct 输出特征图形状：", ct_output.shape)  # torch.Size([1, 1, 4, 4])（2倍上采样）
# print("ct 输出特征图数值：\n", ct_output.detach().squeeze().numpy())

# print("c 输出特征图形状：", c_output.shape) 
# print("c 输出特征图数值：\n", c_output.detach().squeeze().numpy())